<a href="https://colab.research.google.com/github/silvjunu/devowel/blob/main/timbre_bridge_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Timbre Bridge

## Research idea

This project studies whether the timbre of one sound source can be continuously transformed into the timbre of another while preserving as much of the underlying musical content as possible.

Examples:

- instrument A sustained note → instrument B sustained note
- instrument A melody → the same melody in instrument B timbre
- human voice → instrument
- instrument → human-voice-like timbre
- pitched sound → noise-like sound
- white noise → increasingly pitched / instrument-like sound

The central question is not merely whether two recordings can be crossfaded.

The goal is to separate, as far as possible:

**content**

- fundamental frequency / pitch trajectory,
- timing,
- note sequence,
- loudness / dynamics,

from:

**timbre**

- harmonic distribution,
- spectral envelope,
- inharmonic components,
- noise spectrum,
- transient structure,
- periodicity.

Then we can study a continuous morph parameter:

`alpha = 0` → timbre A

`alpha = 1` → timbre B

and eventually:

`alpha(t)` → time-varying timbre transformation.

This notebook begins with a deliberately small DSP-only proof of concept before moving to recorded instruments and more complex models.



## 1. Why This Is Plausible

Timbre transfer and timbre morphing are established research areas.

However, "timbre" is not one single variable.

For a sustained pitched tone, much of the audible difference may be represented by the amplitudes and phases of harmonics plus a residual/noise component.

For complete musical phrases, additional structure matters:

- pitch trajectory,
- loudness trajectory,
- articulation,
- attack and release,
- transients,
- vibrato,
- inharmonicity,
- stochastic/noise components.

Therefore this project will not assume that a single spectral-envelope interpolation solves the whole problem.

We will begin with the easiest controlled case:

**same pitch + same duration + sustained harmonic sounds**

and gradually relax those assumptions.



## 2. Proposed Research Layers

### Layer 1 — Musical content

Represent what should remain approximately invariant:

- `f0(t)` — fundamental-frequency trajectory,
- `loudness(t)` — amplitude / loudness trajectory,
- note timing and duration.

### Layer 2 — Timbre representation

Represent what may change between source A and source B:

- harmonic amplitudes,
- spectral envelope,
- harmonic vs noise balance,
- inharmonicity,
- residual/noise spectrum,
- transient structure.

### Layer 3 — Morph operator

Define a continuous interpolation:

`T(alpha)`

where:

- `alpha = 0` gives timbre A,
- `alpha = 1` gives timbre B,
- intermediate alpha values create timbral states between them.

Later, `alpha` can become time-dependent:

`alpha(t)`

which turns timbre morphing into a synthesizer control signal.



## 3. Project Roadmap

**Phase 0 — Synthetic harmonic proof of concept**

Create two artificial harmonic timbres with identical pitch and duration, then interpolate only their harmonic distributions.

**Phase 1 — Real sustained-note pair**

Record or obtain instrument A and B playing the same sustained pitch. Compare FFT spectra, harmonic amplitudes, spectral envelopes, and residual components.

**Phase 2 — Harmonic + stochastic decomposition**

Separate periodic/harmonic energy from noise-like or residual energy and morph both components independently.

**Phase 3 — Same melody, aligned performances**

Use frame-wise pitch and loudness trajectories while transforming timbre over time.

**Phase 4 — Performance-independent timbre transfer**

Preserve the musical content of one performance while rendering it with another source's learned or estimated timbre.

**Phase 5 — Generalized sources**

Investigate transitions between:

- voice,
- instruments,
- inharmonic sounds,
- noise.

This will likely require an explicit periodicity / harmonic-noise representation rather than a single spectral-envelope model.

**Phase 6 — Real-time synthesizer**

Expose timbre position as a controllable parameter through automation, MIDI, or an XY / latent-space controller.



## 4. Colab Session Setup

Google Drive is used as persistent storage.

The actual DSP workspace is local to the Colab runtime.

This follows the same storage principle that proved robust in the DeVowel project:

- Drive = persistent source/results storage
- `/content` = active DSP workspace


In [ ]:

from google.colab import drive
from pathlib import Path

import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
import librosa

from IPython.display import Audio, display


def find_or_mount_drive():
  candidates = [
    Path("/content/drive"),
    Path("/content/gdrive"),
    Path("/content/timbre_bridge_drive"),
  ]


  for candidate in candidates:
    try:
      if (
        candidate
        / "MyDrive"
      ).is_dir():
        print(
          "Using mounted Google Drive:",
          candidate,
        )

        return candidate

    except OSError:
      pass


  suffix = 0


  while True:
    if suffix == 0:
      mount_point = Path(
        "/content/timbre_bridge_drive"
      )

    else:
      mount_point = Path(
        f"/content/timbre_bridge_drive_{suffix}"
      )


    mount_point.mkdir(
      parents=True,
      exist_ok=True,
    )


    try:
      is_empty = not any(
        mount_point.iterdir()
      )

    except OSError:
      is_empty = False


    if is_empty:
      break


    suffix += 1


  drive.mount(
    str(mount_point)
  )


  if not (
    mount_point
    / "MyDrive"
  ).is_dir():
    raise RuntimeError(
      "Google Drive was mounted, "
      "but MyDrive could not be found."
    )


  return mount_point


drive_root = find_or_mount_drive()

mydrive_dir = (
  drive_root
  / "MyDrive"
)


project_dir = (
  mydrive_dir
  / "timbre_bridge"
)


data_dir = (
  project_dir
  / "data"
)


source_a_dir = (
  data_dir
  / "source_a"
)


source_b_dir = (
  data_dir
  / "source_b"
)


results_dir = (
  project_dir
  / "results"
)


runtime_dir = Path(
  "/content/timbre_bridge_runtime"
)


if runtime_dir.exists():
  shutil.rmtree(
    runtime_dir
  )


runtime_dir.mkdir(
  parents=True,
  exist_ok=True,
)


for directory in [
  project_dir,
  data_dir,
  source_a_dir,
  source_b_dir,
  results_dir,
]:
  directory.mkdir(
    parents=True,
    exist_ok=True,
  )


print()
print(
  "Project directory:",
  project_dir,
)

print(
  "Source A directory:",
  source_a_dir,
)

print(
  "Source B directory:",
  source_b_dir,
)

print(
  "Results directory:",
  results_dir,
)

print(
  "Runtime directory:",
  runtime_dir,
)



## 5. Phase 0 — Synthetic Harmonic Timbres

Before using recorded instruments, we verify the core morphing mechanism with controlled signals.

Both synthetic sounds will have:

- the same fundamental frequency,
- the same duration,
- the same harmonic frequencies,
- the same phase convention.

Only the **harmonic amplitudes** will differ.

This isolates one component of timbre.

Important limitation:

This is not yet "instrument A → instrument B."

It is a controlled demonstration that a reusable timbre representation can be interpolated continuously while pitch and timing remain fixed.


In [ ]:

def synthesize_harmonic_timbre(
  f0,
  sr,
  duration,
  harmonic_amps,
):
  t = np.arange(
    int(
      sr
      * duration
    )
  ) / sr


  wave = np.zeros_like(
    t
  )


  for harmonic_number, amp in enumerate(
    harmonic_amps,
    start=1,
  ):
    freq = (
      harmonic_number
      * f0
    )


    if freq >= sr / 2:
      break


    wave += (
      amp
      * np.sin(
        2
        * np.pi
        * freq
        * t
      )
    )


  peak = np.max(
    np.abs(
      wave
    )
  )


  if peak > 0:
    wave = (
      0.8
      * wave
      / peak
    )


  return wave


def interpolate_harmonic_amps_db(
  amps_a,
  amps_b,
  alpha,
  floor_db=-80,
):
  amps_a = np.asarray(
    amps_a,
    dtype=float,
  )


  amps_b = np.asarray(
    amps_b,
    dtype=float,
  )


  if len(amps_a) != len(
    amps_b
  ):
    raise ValueError(
      "amps_a and amps_b must have "
      "the same length."
    )


  floor_amp = (
    10
    ** (
      floor_db
      / 20
    )
  )


  db_a = (
    20
    * np.log10(
      np.maximum(
        amps_a,
        floor_amp,
      )
    )
  )


  db_b = (
    20
    * np.log10(
      np.maximum(
        amps_b,
        floor_amp,
      )
    )
  )


  morph_db = (
    (1 - alpha)
    * db_a
    + alpha
    * db_b
  )


  morph_amps = (
    10
    ** (
      morph_db
      / 20
    )
  )


  return morph_amps



### Synthetic timbre definitions

These are intentionally artificial.

Timbre A has a relatively smooth harmonic decay.

Timbre B has a more uneven, hollow / odd-rich distribution.

They are not intended to imitate specific real instruments.

The goal is only to make the timbral transformation easy to hear.


In [ ]:

sr = 44100
duration = 1.5
f0 = 220.0


harmonic_amps_a = np.array([
  1.00,
  0.72,
  0.52,
  0.38,
  0.28,
  0.20,
  0.15,
  0.11,
  0.08,
  0.06,
])


harmonic_amps_b = np.array([
  1.00,
  0.12,
  0.82,
  0.10,
  0.58,
  0.08,
  0.40,
  0.06,
  0.28,
  0.05,
])


harmonic_numbers = np.arange(
  1,
  len(
    harmonic_amps_a
  )
  + 1,
)


plt.figure(
  figsize=(10, 4)
)


plt.stem(
  harmonic_numbers,
  harmonic_amps_a,
  label="Timbre A",
)


plt.stem(
  harmonic_numbers,
  harmonic_amps_b,
  label="Timbre B",
)


plt.xlabel(
  "Harmonic number"
)

plt.ylabel(
  "Amplitude"
)

plt.title(
  "Synthetic Harmonic Timbres"
)

plt.legend()

plt.show()



## 6. Static Timbre Morph

Five interpolation positions are generated:

- `alpha = 0.00` → timbre A
- `alpha = 0.25`
- `alpha = 0.50`
- `alpha = 0.75`
- `alpha = 1.00` → timbre B

The interpolation is performed in dB rather than directly in linear amplitude.

This is not automatically the "correct" morphing domain; later experiments will compare alternative representations.

For now it provides a simple first baseline.


In [ ]:

morph_alphas = [
  0.00,
  0.25,
  0.50,
  0.75,
  1.00,
]


synthetic_morphs = {}


for alpha in morph_alphas:
  morph_amps = interpolate_harmonic_amps_db(
    harmonic_amps_a,
    harmonic_amps_b,
    alpha,
  )


  morph_wave = synthesize_harmonic_timbre(
    f0,
    sr,
    duration,
    morph_amps,
  )


  synthetic_morphs[
    alpha
  ] = (
    morph_wave,
    morph_amps,
  )


for index, alpha in enumerate(
  morph_alphas,
  start=1,
):
  morph_wave, _ = synthetic_morphs[
    alpha
  ]


  print()
  print(
    f"{index}. Synthetic timbre morph — "
    f"alpha = {alpha:.2f}"
  )


  print(
    "Same pitch and duration; "
    "only the harmonic distribution changes."
  )


  display(
    Audio(
      morph_wave,
      rate=sr,
    )
  )



## 7. Dynamic Timbre Morph

A static alpha creates one fixed timbre.

A time-dependent control signal `alpha(t)` creates a timbre that evolves within one sound.

The next cell performs a continuous A → B transformation while preserving the same fundamental frequency.

This is the simplest prototype of a timbre-morphing synthesizer control.


In [ ]:

def synthesize_dynamic_harmonic_morph(
  f0,
  sr,
  duration,
  amps_a,
  amps_b,
):
  n_samples = int(
    sr
    * duration
  )


  t = np.arange(
    n_samples
  ) / sr


  alpha_t = np.linspace(
    0.0,
    1.0,
    n_samples,
  )


  wave = np.zeros(
    n_samples
  )


  floor_amp = (
    10
    ** (
      -80
      / 20
    )
  )


  db_a = (
    20
    * np.log10(
      np.maximum(
        amps_a,
        floor_amp,
      )
    )
  )


  db_b = (
    20
    * np.log10(
      np.maximum(
        amps_b,
        floor_amp,
      )
    )
  )


  for harmonic_number in range(
    1,
    len(amps_a) + 1,
  ):
    freq = (
      harmonic_number
      * f0
    )


    if freq >= sr / 2:
      break


    amp_db_t = (
      (1 - alpha_t)
      * db_a[
        harmonic_number - 1
      ]
      + alpha_t
      * db_b[
        harmonic_number - 1
      ]
    )


    amp_t = (
      10
      ** (
        amp_db_t
        / 20
      )
    )


    wave += (
      amp_t
      * np.sin(
        2
        * np.pi
        * freq
        * t
      )
    )


  peak = np.max(
    np.abs(
      wave
    )
  )


  if peak > 0:
    wave = (
      0.8
      * wave
      / peak
    )


  return (
    wave,
    alpha_t,
  )


dynamic_wave, dynamic_alpha = (
  synthesize_dynamic_harmonic_morph(
    f0,
    sr,
    duration,
    harmonic_amps_a,
    harmonic_amps_b,
  )
)


print(
  "1. Dynamic synthetic timbre A → B"
)

print(
  "The pitch remains fixed while the "
  "harmonic distribution changes continuously."
)


display(
  Audio(
    dynamic_wave,
    rate=sr,
  )
)


In [ ]:

time_axis = np.arange(
  len(
    dynamic_alpha
  )
) / sr


plt.figure(
  figsize=(10, 4)
)


plt.plot(
  time_axis,
  dynamic_alpha,
)


plt.xlabel(
  "Time (s)"
)

plt.ylabel(
  "Timbre alpha"
)

plt.title(
  "Dynamic Timbre Control"
)

plt.ylim(
  -0.05,
  1.05,
)

plt.grid(
  alpha=0.2,
)


plt.show()



## 8. First Real-Recording Experiment

The next research step should use two real sustained tones.

Recommended first dataset:

- one monophonic source A,
- one monophonic source B,
- approximately the same fundamental frequency,
- approximately the same stable duration,
- minimal room noise and reverberation,
- preferably little vibrato for the first experiment.

The first real experiment should **not** immediately use a full melody.

We should first determine which representation actually produces a convincing A ↔ B morph for sustained sounds.

Candidate representations to compare:

1. raw magnitude-spectrum interpolation,
2. smoothed spectral-envelope interpolation,
3. harmonic-amplitude interpolation,
4. harmonic + residual/noise decomposition,
5. later, sinusoidal-model or DDSP-style representations.

The result of Phase 1 will determine how the melody experiment should be designed.



## 9. Methodological Rule

A crossfade is not a timbre morph.

Waveform crossfading:

`(1 - alpha) x_A(t) + alpha x_B(t)`

simply mixes two signals.

This project instead aims to transform a **representation of timbre** while preserving a representation of musical content.

Therefore every experiment should explicitly state:

- what is treated as content,
- what is treated as timbre,
- what representation is being interpolated,
- what information is intentionally held constant,
- what information is allowed to change.

This distinction is central to the project.



## 10. Research Checkpoint

At the start of the project:

**Established**

- Timbre transfer and timbre morphing are existing research areas.
- Sustained harmonic sounds can be represented using harmonic/sinusoidal and spectral components.
- DDSP-style systems explicitly separate control features such as pitch and loudness from learned timbral synthesis parameters.

**Project inference**

- The DeVowel result suggests that reusable spectral deviations can sometimes act as effective synthesis controls.
- A related decomposition may be useful for instrument-to-instrument timbre morphing.

**Open hypotheses**

- A compact reusable timbre representation may interpolate convincingly between very different sound sources.
- Harmonic/noise balance may provide a useful bridge between pitched instruments and noise-like sources.
- A continuous timbre path may be musically useful even when it does not correspond to a physically realizable acoustic instrument.
